In [ ]:
'''
Clip a global or large-scale GeoTIFF raster (e.g., a netgain raster) into separate smaller files based on multiple regions. stream the cropped raster out chunk by chunk, apply a geometry mask to keep only the target region. (e.g., “Europe,” “Africa,” “South America,” etc.).
'''
import os
import glob
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import rasterio
from rasterio import mask as rio_mask
from rasterio.windows import Window
from shapely.geometry import shape, box

INPUT_DIR = os.getenv("INPUT_DIR", "./data/tif") # download from "1.Download_tifdata.ipynb"
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "./outputs/regions_final")

NUM_CORES = int(os.getenv("NUM_CORES", "20"))
MAX_WORKERS = int(os.getenv("MAX_WORKERS", str(max(1, NUM_CORES - 1))))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1024"))
PADDING = int(os.getenv("PADDING", "2"))


REGIONS = [
    {
        "name": "South_America",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-31.57, 14.10], [-72.14, 14.10], [-82.10, 0.0],
                [-180.0, 0.0], [-180.0, -66.34], [-31.57, -66.34],
                [-31.57, 14.10]
            ]]
        }
    },
    {
        "name": "North_American",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-180.0, 66.34], [-180.0, 0.0], [-82.10, 0.0],
                [-72.14, 14.10], [-31.57, 14.10], [-31.57, 66.34],
                [-180.0, 66.34]
            ]]
        }
    },
    {
        "name": "Europe",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-31.57, 66.34],
                [-31.57, 37.524],
                [11.178, 37.524],
                [32.196, 30.910],
                [39.869, 48.048],
                [29.236, 66.34]
            ]]
        }
    },
    {
        "name": "Africa",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-31.57, -66.34],
                [52.64, -66.34],
                [52.64, 14.092],
                [43.594, 12.591],
                [32.196, 30.910],
                [11.178, 37.524],
                [-31.57, 37.524],
                [-31.57, -66.34]
            ]]
        }
    },
    {
        "name": "South_East_Asia",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [88.837, 17.176],
                [97.568, 28.548],
                [108.1, 21.5],
                [141.019, 21.5],
                [141.019, -10.5],
                [88.387, -10.5],
                [88.837, 17.176]
            ]]
        }
    },
    {
        "name": "Australia",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [52.64, -10.5],
                [52.64, -66.34],
                [180.0, -66.34],
                [180.0, 0.0],
                [141.019, 0.0],
                [141.019, -10.5],
                [52.64, -10.5]
            ]]
        }
    },
    {
        "name": "Asia_noEAS",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [29.236, 66.34],
                [39.869, 48.048],
                [32.196, 30.910],
                [43.594, 12.591],
                [52.64, 14.092],
                [52.64, -10.5],
                [88.387, -10.5],
                [88.837, 17.176],
                [97.568, 28.548],
                [108.1, 21.5],
                [141.019, 21.5],
                [141.019, 0.0],
                [180.0, 0.0],
                [180.0, 66.34]
            ]]
        }
    }
]


def process_region(region, src, output_dir, padding, chunk_size, file_id):
    """
    Clip one raster against one region and write the result incrementally.
    """
    try:
        geom = shape(region["geometry"])

        # Repair invalid geometry if needed.
        if not geom.is_valid:
            geom = geom.buffer(0)

        # Restrict the region to the raster footprint.
        raster_poly = box(*src.bounds)
        geom = geom.intersection(raster_poly)

        if geom.is_empty:
            return False, f"{region['name']} has no overlap with the raster"

        # Compute a padded crop window in raster coordinates.
        res_x, res_y = abs(src.res[0]), abs(src.res[1])
        minx, miny, maxx, maxy = geom.bounds

        adj_minx = minx - padding * res_x
        adj_maxx = maxx + padding * res_x
        adj_miny = miny - padding * res_y
        adj_maxy = maxy + padding * res_y

        row_start, col_start = src.index(adj_minx, adj_maxy)
        row_end, col_end = src.index(adj_maxx, adj_miny)

        row_start = max(0, row_start)
        col_start = max(0, col_start)
        row_end = min(src.height, row_end + 1)
        col_end = min(src.width, col_end + 1)

        width = col_end - col_start
        height = row_end - row_start

        if width <= 0 or height <= 0:
            return False, (
                f"{region['name']} has invalid window size: "
                f"width={width}, height={height}"
            )

        meta = src.meta.copy()
        meta.update({
            "driver": "GTiff",
            "height": height,
            "width": width,
            "transform": src.window_transform(Window(col_start, row_start, width, height)),
            "nodata": src.nodata if src.nodata is not None else 0,
            "compress": "lzw",
            "tiled": True,
        })

        out_name = f"{region['name'].lower()}_{file_id}.tif"
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            # Stream the output row blocks to keep memory usage low.
            for y0 in range(row_start, row_end, chunk_size):
                h = min(chunk_size, row_end - y0)
                src_window = Window(col_start, y0, width, h)

                data = src.read(1, window=src_window)

                # Build a geometry mask for the current chunk only.
                mask_arr = rio_mask.geometry_mask(
                    [geom],
                    out_shape=data.shape,
                    transform=src.window_transform(src_window),
                    invert=True
                )

                clipped = np.where(mask_arr, data, meta["nodata"]).astype(src.dtypes[0])

                dst_window = Window(0, y0 - row_start, width, h)
                dst.write(clipped, 1, window=dst_window)

        return True, out_path

    except Exception:
        return False, traceback.format_exc()

def process_task(task):
    """
    Open one raster independently inside the worker and process one region.
    """
    tif_path, region, file_id = task
    try:
        with rasterio.open(tif_path) as src:
            return process_region(
                region=region,
                src=src,
                output_dir=OUTPUT_DIR,
                padding=PADDING,
                chunk_size=CHUNK_SIZE,
                file_id=file_id
            )
    except Exception:
        return False, traceback.format_exc()


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    tif_list = glob.glob(os.path.join(INPUT_DIR, "*.tif"))
    tasks = []

    for tif_path in tif_list:
        # some_name_07.tif -> file_id = "07"
        file_id = os.path.splitext(os.path.basename(tif_path))[0].split("_")[-1]
        for region in REGIONS:
            tasks.append((tif_path, region, file_id))

    total = len(tasks)
    success = 0

    print(f"Submitting {total} tasks with {MAX_WORKERS} worker threads...")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_task, task): task for task in tasks}

        for idx, fut in enumerate(as_completed(futures), start=1):
            ok, msg = fut.result()
            if ok:
                success += 1
                print(f"[{idx}/{total}] [OK] {os.path.basename(msg)}")
            else:
                print(f"[{idx}/{total}] [FAILED] {msg}")

    print(f"Completed: {success}/{total} outputs generated successfully")


if __name__ == "__main__":
    main()

In [ ]:
"""Split region-level loss/gain rasters by year code and build netgain rasters.

Configuration names are aligned with the first script where practical:
- INPUT_DIR
- OUTPUT_DIR
- CHUNK_SIZE
"""

from __future__ import annotations

import logging
import os
from contextlib import ExitStack
from pathlib import Path
from typing import Iterator

import numpy as np
import rasterio
from rasterio.windows import Window

LOGGER = logging.getLogger(__name__)

INPUT_DIR = Path(os.getenv("INPUT_DIR", "./data/region_mosaic"))
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "./outputs/region_mosaic_processed"))
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "1024"))

YEAR_CODES = [4, 7, 10, 13, 16, 19]
YEAR_SUFFIXES = [f"{code:02d}" for code in YEAR_CODES]

BACKGROUND = int(os.getenv("BACKGROUND", "0"))
NET_NODATA = int(os.getenv("NET_NODATA", "0"))
NET_DTYPE = np.int8  # valid values: -1, 0, 1

SPLIT_BASE_DIR = OUTPUT_DIR / "split_by_year_suffix"
SPLIT_LOSS_DIR = SPLIT_BASE_DIR / "lossYear"
SPLIT_GAIN_DIR = SPLIT_BASE_DIR / "gainYear"
NETGAIN_DIR = OUTPUT_DIR / "netgain_from_split"


def configure_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )


def ensure_output_dirs() -> None:
    SPLIT_LOSS_DIR.mkdir(parents=True, exist_ok=True)
    SPLIT_GAIN_DIR.mkdir(parents=True, exist_ok=True)
    NETGAIN_DIR.mkdir(parents=True, exist_ok=True)


def parse_region_from_filename(file_path: Path) -> str:
    """Extract region name from files like lossYear_Region.tif or gainYear_Region.tif."""
    stem = file_path.stem
    return stem.replace("lossYear_", "").replace("gainYear_", "")


def iter_windows(height: int, width: int, chunk_size: int) -> Iterator[Window]:
    """Yield raster windows for chunked reading/writing."""
    for row in range(0, height, chunk_size):
        window_height = min(chunk_size, height - row)
        for col in range(0, width, chunk_size):
            window_width = min(chunk_size, width - col)
            yield Window(col, row, window_width, window_height)


def split_one_raster(src_path: Path, output_dir: Path) -> None:
    """Split one region raster into multiple rasters by target year code."""
    stem = src_path.stem
    expected_outputs = [output_dir / f"{stem}_{code:02d}.tif" for code in YEAR_CODES]

    if all(path.exists() for path in expected_outputs):
        LOGGER.info("[SPLIT-SKIP] %s", stem)
        return

    LOGGER.info("[SPLIT] %s", stem)

    with rasterio.open(src_path) as src:
        if src.count != 1:
            raise ValueError(f"{src_path} must be single-band, found {src.count} band(s)")

        profile = src.profile.copy()
        profile.update(count=1, nodata=BACKGROUND, compress="LZW")

        with ExitStack() as stack:
            destinations = {
                code: stack.enter_context(
                    rasterio.open(output_dir / f"{stem}_{code:02d}.tif", "w", **profile)
                )
                for code in YEAR_CODES
            }

            for window in iter_windows(src.height, src.width, CHUNK_SIZE):
                src_array = src.read(1, window=window)

                for code, dst in destinations.items():
                    out_array = np.zeros(src_array.shape, dtype=src_array.dtype)
                    out_array[src_array == code] = code
                    dst.write(out_array, 1, window=window)


def validate_alignment(src_loss: rasterio.io.DatasetReader, src_gain: rasterio.io.DatasetReader) -> None:
    """Validate that loss/gain rasters are spatially aligned."""
    if src_loss.shape != src_gain.shape:
        raise ValueError("Loss and gain rasters have different shapes")
    if src_loss.transform != src_gain.transform:
        raise ValueError("Loss and gain rasters have different transforms")
    if src_loss.crs != src_gain.crs:
        raise ValueError("Loss and gain rasters have different CRS values")


def build_netgain_for_region_suffix(region: str, suffix: str) -> None:
    """Build one netgain raster for a specific region and year suffix."""
    loss_path = SPLIT_LOSS_DIR / f"lossYear_{region}_{suffix}.tif"
    gain_path = SPLIT_GAIN_DIR / f"gainYear_{region}_{suffix}.tif"
    out_path = NETGAIN_DIR / f"netgain_{region}_{suffix}.tif"

    if not loss_path.exists() and not gain_path.exists():
        return

    if out_path.exists():
        LOGGER.info("[NET-SKIP] %s", out_path.name)
        return

    ref_path = loss_path if loss_path.exists() else gain_path

    with rasterio.open(ref_path) as ref:
        profile = ref.profile.copy()
        profile.update(dtype=rasterio.int8, count=1, nodata=NET_NODATA, compress="LZW")

        with ExitStack() as stack:
            src_loss = stack.enter_context(rasterio.open(loss_path)) if loss_path.exists() else None
            src_gain = stack.enter_context(rasterio.open(gain_path)) if gain_path.exists() else None

            if src_loss and src_gain:
                validate_alignment(src_loss, src_gain)

            with rasterio.open(out_path, "w", **profile) as dst:
                for window in iter_windows(ref.height, ref.width, CHUNK_SIZE):
                    net_array = np.zeros((int(window.height), int(window.width)), dtype=NET_DTYPE)

                    if src_loss is not None:
                        loss_array = src_loss.read(1, window=window)
                        net_array[loss_array > 0] -= 1

                    if src_gain is not None:
                        gain_array = src_gain.read(1, window=window)
                        net_array[gain_array > 0] += 1

                    dst.write(net_array, 1, window=window)

    LOGGER.info("[NET] %s", out_path.name)


def main() -> None:
    configure_logging()
    ensure_output_dirs()

    loss_source_files = sorted(INPUT_DIR.glob("lossYear_*.tif"))
    gain_source_files = sorted(INPUT_DIR.glob("gainYear_*.tif"))

    LOGGER.info("Found loss source rasters: %d", len(loss_source_files))
    LOGGER.info("Found gain source rasters: %d", len(gain_source_files))

    for src_path in loss_source_files:
        split_one_raster(src_path, SPLIT_LOSS_DIR)

    for src_path in gain_source_files:
        split_one_raster(src_path, SPLIT_GAIN_DIR)

    regions = sorted(
        {
            *(parse_region_from_filename(path) for path in loss_source_files),
            *(parse_region_from_filename(path) for path in gain_source_files),
        }
    )

    LOGGER.info("Regions: %s", ", ".join(regions) if regions else "<none>")
    LOGGER.info("Year suffixes: %s", ", ".join(YEAR_SUFFIXES))

    for region in regions:
        for suffix in YEAR_SUFFIXES:
            build_netgain_for_region_suffix(region, suffix)

    LOGGER.info("Completed. Split outputs: %s", SPLIT_BASE_DIR)
    LOGGER.info("Completed. Netgain outputs: %s", NETGAIN_DIR)


if __name__ == "__main__":
    main()
